# OCR quantitative evaluation: raw OCR vs LLM post-processing

This notebook presents a quantitative evaluation of the trained OCR model, both before and after LLM-based post-processing.

The model was trained on a dataset derived from historical Spanish prints. Details on dataset preparation can be found in [scripts/README.md](../scripts/README.md), while information about the training procedure is provided in [README.md](../README.md). The dataset was split such that five prints were used for training and one was held out for validation. In this notebook, we evaluate the model quantitatively on this validation set.

The OCR model considered here focuses exclusively on text recognition from already segmented line images. However, the complete pipeline developed in this project also supports full-page transcription by incorporating a line segmentation step using external tools. For a full description of the inference pipeline, refer to [inference.ipynb](./inference.ipynb).

## Necessary imports to run this notebook

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR.resolve()) not in sys.path:
    sys.path.append(str(SRC_DIR.resolve()))

In [2]:
import csv
import math
import re
from collections import defaultdict
from pathlib import Path
from types import SimpleNamespace
import pandas as pd
import torch
from torch.utils.data import DataLoader
from Levenshtein import distance as levenshtein_distance
from dataset import CsvLineDataset, AlignCollate
from model import Model
from utils import CTCLabelConverter
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
from difflib import SequenceMatcher
from collections import Counter
from IPython.display import display, HTML
import html
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

/home/tomas/projects/renAIssance/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

As explained at [README.md](../README.md), the model was trained on five documents and validated on only one of them. For validation, **Covarrubias – *Tesoro de la lengua*** was chosen.

In [3]:
# --- Required paths ---
DATASET_CSV = PROJECT_ROOT / "data" / "filtered_spanish_dataset_ordered.csv"
IMAGE_ROOT = PROJECT_ROOT / "data"
CHECKPOINT_PATH = PROJECT_ROOT / "models" / "OCR_model_v2.ckpt"

# --- Select the documents to evaluate
PDF_IDS = [
    "Covarrubias - Tesoro lengua"
]

# --- Dataloader settings ---
BATCH_SIZE = 8
NUM_WORKERS = 0

# --- Output directory ---
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "evaluation_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Helper functions

This section defines several utility functions used throughout the notebook, including:
- Loading and filtering evaluation data  
- Loading pretrained model weights  
- Extracting predicted text from model outputs  
- Computing evaluation metrics  

In [4]:
def read_dataset_rows(dataset_csv):
    rows = []
    with open(dataset_csv, "r", newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            if row.get("crop_rel") and row.get("gt_text") is not None and row.get("pdf_id"):
                rows.append(row)
    return rows


def filter_rows_by_pdf_ids(rows, pdf_ids):
    pdf_id_set = set(pdf_ids)
    return [row for row in rows if row["pdf_id"] in pdf_id_set]


def normalize_text(text, strip_extra_spaces=False):
    text = "" if text is None else str(text)
    if strip_extra_spaces:
        text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_global_text(text, lowercase=False, remove_eol_hyphen=True):
    text = "" if text is None else str(text)

    if lowercase:
        text = text.lower()

    # Remove hyphenation caused by line breaks: "pala-\nbra" -> "palabra"
    if remove_eol_hyphen:
        text = re.sub(r'-\s*\n\s*', '', text)

    # Replace remaining line breaks with spaces
    text = re.sub(r'\s*\n\s*', ' ', text)

    # Collapse repeated whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def load_model_from_checkpoint(ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location=device)

    hparams = ckpt["hyper_parameters"]
    opt = SimpleNamespace(**hparams)
    opt.input_channel = 1

    converter = CTCLabelConverter(opt.character)
    opt.num_class = len(converter.character)

    state_dict = ckpt["state_dict"]
    state_dict = {k.replace("model.", ""): v for k, v in state_dict.items()}

    model = Model(opt)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    return model, opt, converter


def word_level_distance(pred, ref):
    pred_words = pred.split()
    ref_words = ref.split()

    if len(ref_words) == 0:
        return 0 if len(pred_words) == 0 else len(pred_words)

    dp = [[0] * (len(ref_words) + 1) for _ in range(len(pred_words) + 1)]
    for i in range(len(pred_words) + 1):
        dp[i][0] = i
    for j in range(len(ref_words) + 1):
        dp[0][j] = j

    for i in range(1, len(pred_words) + 1):
        for j in range(1, len(ref_words) + 1):
            cost = 0 if pred_words[i - 1] == ref_words[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost,  # substitution
            )

    return dp[-1][-1]


def compute_metrics(predictions, references, strip_extra_spaces=False):
    preds = [
        normalize_text(p, strip_extra_spaces=strip_extra_spaces)
        for p in predictions
    ]
    refs = [
        normalize_text(r, strip_extra_spaces=strip_extra_spaces)
        for r in references
    ]

    total_edit_distance = 0
    total_ref_chars = 0
    total_word_distance = 0
    total_ref_words = 0
    ned_sum = 0.0

    for pred, ref in zip(preds, refs):
        ed = levenshtein_distance(pred, ref)
        total_edit_distance += ed
        total_ref_chars += len(ref)

        wd = word_level_distance(pred, ref)
        total_word_distance += wd
        total_ref_words += len(ref.split())

        denom = max(len(pred), len(ref), 1)
        ned_sum += ed / denom

    cer = total_edit_distance / max(total_ref_chars, 1)
    wer = total_word_distance / max(total_ref_words, 1)
    avg_ned = ned_sum / max(len(refs), 1)
    char_accuracy = 1.0 - cer

    return {
        "num_samples": len(refs),
        "cer": cer,
        "char_accuracy": char_accuracy,
        "avg_normalized_edit_distance": avg_ned,
        "wer": wer,
    }


def metrics_to_frame(name, metrics):
    return pd.DataFrame(
        [{
            "system": name,
            "num_samples": metrics["num_samples"],
            "CER": metrics["cer"],
            "Char accuracy": metrics["char_accuracy"],
            "Avg. normalized edit distance": metrics["avg_normalized_edit_distance"],
            "WER": metrics["wer"],
        }]
    )


def print_metrics(title, metrics, show_ned=True):
    print(title)
    print("-" * len(title))
    print(f"Samples:                    {metrics['num_samples']}")
    print(f"CER:                        {metrics['cer']:.4f}")
    print(f"Character accuracy:         {metrics['char_accuracy']:.4f}")
    if show_ned:
        print(f"Avg. normalized ED:         {metrics['avg_normalized_edit_distance']:.4f}")
    print(f"WER:                        {metrics['wer']:.4f}")


def extract_prediction_texts(decoded_preds):
    return [str(p) for p in decoded_preds]


def add_error_columns(df, pred_col, gt_col="ground_truth"):
    out = df.copy()
    out["char_edit_distance"] = [
        levenshtein_distance(str(p), str(g))
        for p, g in zip(out[pred_col], out[gt_col])
    ]
    out["ref_length"] = out[gt_col].astype(str).str.len()
    out["exact_match"] = (out[pred_col].astype(str) == out[gt_col].astype(str))
    return out


def tfidf_cosine_similarity(text1: str, text2: str) -> float:
    """
    Compute cosine similarity between two texts using TF-IDF vectors.
    Returns a value in [0, 1].
    """
    if not isinstance(text1, str) or not isinstance(text2, str):
        raise ValueError("Both inputs must be strings.")

    if not text1.strip() or not text2.strip():
        return 0.0

    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([text1, text2])
    sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])
    return float(sim[0][0])


def jaccard_similarity(text1: str, text2: str) -> float:
    """
    Compute word-level Jaccard similarity between two texts.
    Returns a value in [0, 1].
    """
    if not isinstance(text1, str) or not isinstance(text2, str):
        raise ValueError("Both inputs must be strings.")

    words1 = set(text1.split())
    words2 = set(text2.split())

    if not words1 and not words2:
        return 1.0
    if not words1 or not words2:
        return 0.0

    return len(words1 & words2) / len(words1 | words2)

## Load model and selected rows

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading checkpoint from: {CHECKPOINT_PATH}")
model, opt, converter = load_model_from_checkpoint(CHECKPOINT_PATH, device)

print(f"Reading dataset from: {DATASET_CSV}")
rows = read_dataset_rows(DATASET_CSV)
rows = filter_rows_by_pdf_ids(rows, PDF_IDS)

print(f"Selected pdf_ids: {PDF_IDS}")
print(f"Rows before dataset filtering: {len(rows)}")

Loading checkpoint from: /home/tomas/projects/renAIssance/models/OCR_model_v2.ckpt
Reading dataset from: /home/tomas/projects/renAIssance/data/filtered_spanish_dataset_ordered.csv
Selected pdf_ids: ['Covarrubias - Tesoro lengua']
Rows before dataset filtering: 113


## Build dataset and dataloader

In [6]:
dataset = CsvLineDataset(rows, opt=opt, image_root=IMAGE_ROOT, is_train=False)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=AlignCollate(imgH=opt.imgH),
)

## Run raw OCR evaluation

Here, I run raw OCR evaluation. That is, we compute directly the output provided by the model and evaluate it against the ground truth text extracted from the provided `.docx` files. As mentioned in [scripts/README.md](../scripts/README.md), the ground truth contains several inconsistencies and transcription errors. These flaws may introduce noise during both training and evaluation. Cleaning the ground truth data was out of the scope of this project; however, the obtained metrics still provide useful insights into the performance of the proposed methodology.

### Evaluation metrics

The following metrics are used to quantitatively assess OCR performance:

---

**Character Error Rate (CER)**  
Measures the proportion of character-level errors between prediction and ground truth:

$$
CER = \frac{S + D + I}{|y|}
$$

where:
- $S$ is the number of substitutions,
- $D$ is the number of deletions,
- $I$ is the number of insertions,
- and $|y|$ is the number of characters in the ground truth.

Equivalently, this corresponds to the Levenshtein (edit) distance $D(\hat{y}, y)$ between predicted text $\hat{y}$ and ground truth $y$. Lower values indicate better performance.

---

**Character Accuracy**  
Represents the proportion of correctly predicted characters:

$$
\text{Character Accuracy} = 1 - CER
$$

This is often easier to interpret than CER, as it directly reflects correctness.

---

**Word Error Rate (WER)**  
Measures errors at the word level instead of characters:

$$
WER = \frac{D_w(\hat{y}, y)}{N_w}
$$

where $D_w$ is the edit distance computed over words and $N_w$ is the number of words in the ground truth. This metric better reflects readability and semantic correctness.

---

**Average Normalized Edit Distance (NED)**  
Computes edit distance per sample and normalizes by sequence length:

$$
NED = \frac{1}{N} \sum_{i=1}^{N} \frac{D(\hat{y}_i, y_i)}{\max(|\hat{y}_i|, |y_i|, 1)}
$$
---


In [7]:
raw_predictions = []
references = []

with torch.no_grad():
    for images, labels in dataloader:
        images = images.to(device)
        batch_size = images.size(0)

        preds = model(images)
        _, preds_index = preds.max(2)
        preds_size = torch.IntTensor([preds.size(1)] * batch_size)

        preds_str = converter.decode(preds_index, preds_size)

        raw_predictions.extend(extract_prediction_texts(preds_str))
        references.extend([str(x) for x in labels])

raw_metrics = compute_metrics(
    raw_predictions,
    references,
    strip_extra_spaces=True,
)

print_metrics("Raw OCR results", raw_metrics)

Raw OCR results
---------------
Samples:                    109
CER:                        0.1279
Character accuracy:         0.8721
Avg. normalized ED:         0.1391
WER:                        0.4433


### Additionally, we build a table to visualize ground truth text next to OCR predictions.

In [8]:
results_df = pd.DataFrame({
    "ground_truth": references,
    "ocr_prediction": raw_predictions,
})

results_df = add_error_columns(results_df, pred_col="ocr_prediction", gt_col="ground_truth")
results_df.head(20)

,ground_truth,ocr_prediction,char_edit_distance,ref_length,exact_match
0,PEDRO DE VALENCIA CORONISTA,PODOOOE?ALE?o12OOOO?65T2,17,27,False
1,General de el Rey nuestro,Ceneral de el Rey nuestro,1,25,False
2,Señor.,Señor.,0,6,True
3,"POr mandado del Consejo Supremo de Castilla, h...","DOr mandado del Consejo Supremo de Casiilla, h...",2,66,False
4,"titulado, Tesoro de la lengua Castellana, que ...","titulado. Tesoro de la lengua Castellana, que...",2,63,False
5,"ciado Don Sebastian de Covarruvias y Orozco, C...","ciado Don Segastian de Covatruuas y Orozco, Ca...",4,65,False
6,"cuela de la Iglesia de Cuenca, Consultor de el...","cuela de la Iglesia de Cvenca, Consultor de el...",2,74,False
7,"cion, y Capellan de su Magestad, y no he allad...","cion, y Capelan de su Aagestad, y no he hallad...",4,70,False
8,"la Fe, ni a las buenas costumbres; antes tiene...","la Fe, ni a las bvenas costumdres; antes tiene...",3,77,False
9,"no de varia, y curiosa leccion, y doctrina. Po...","mo de varia, y curiosaleccion, y doctrina. Por...",3,76,False


In [9]:
raw_csv_path = OUTPUT_DIR / "ocr_predictions_eval.csv"
results_df.to_csv(raw_csv_path, index=False, encoding="utf-8")
print(f"Saved raw OCR predictions to: {raw_csv_path}")

Saved raw OCR predictions to: /home/tomas/projects/renAIssance/notebooks/evaluation_outputs/ocr_predictions_eval.csv


## LLM-based Post-processing

To further improve the OCR output, we apply a **large language model (LLM)** to post-process the predicted text lines. The LLM is used to correct likely transcription errors by considering the **linguistic context of the sentence**, which can help resolve ambiguities produced by the OCR model.

The post-processing step uses the [Qwen2.5-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct) model, an open-weight language model developed by the Qwen team. The model is executed locally using the [llama.cpp](https://github.com/ggml-org/llama.cpp) inference framework.

The model weights are downloaded from the [Hugging Face Hub](https://huggingface.co/), a widely used platform for sharing machine learning models and datasets.

In this pipeline, the LLM receives the OCR predictions together with a simple system prompt. More complex and detailed prompts were used, including notes taken by experts over the original prints, but for a lighweight model simpler prompts proved to get better results. The model then produces a corrected version of the text while preserving the intended reading of the original document. The system prompt is written in Spanish, as the text being processed is also in that language.

In this implementation, the LLM generates text without imposing formatting constraints relative to the OCR input. Empirically, this approach produced more coherent corrections. On some runs, the output respects the line structure on the input; on others, the LLM produces continuous text. With larger or more capable language models, it may be possible to obtain structured outputs that preserve the original line segmentation while maintaining similar correction quality.

For this reason, we don't use line-to-line evaluation, but only **global text evaluation**.

In [10]:
model_path = hf_hub_download(
    repo_id="Qwen/Qwen2.5-3B-Instruct-GGUF",
    filename="qwen2.5-3b-instruct-q4_k_m.gguf",
)

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=8,
)

llama_model_loader: loaded meta data with 26 key-value pairs and 435 tensors from /home/tomas/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct-GGUF/snapshots/7dabda4d13d513e3e842b20f0d435c732f172cbe/qwen2.5-3b-instruct-q4_k_m.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = qwen2.5-3b-instruct
llama_model_loader: - kv   3:                            general.version str              = v0.1-v0.1
llama_model_loader: - kv   4:                           general.finetune str              = qwen2.5-3b-instruct
llama_model_loader: - kv   5:                         general.size_label str              = 3.4B
llama_model_loade

In [22]:
SYSTEM_PROMPT = """
Corrige el siguiente texto OCR con MUCHOS errores.

Instrucciones:
- Reconstruye el texto como debería leerse en español correcto.
- Corrige palabras completas, no solo letras sueltas.
- Une palabras separadas incorrectamente y separa las que estén pegadas.
- Interpreta palabras deformadas.
- Elimina ruido sin sentido (símbolos, números aleatorios, texto corrupto).
- Mantén nombres propios históricos correctos si se pueden reconocer.
- El resultado debe ser un texto fluido y legible, como un documento real.

IMPORTANTE:
- Es mejor corregir demasiado que quedarse corto.
- No devuelvas texto corrupto sin corregir.

Devuelve SOLO el texto corregido, sin comentarios ni notas adicionales.
"""

### If the text is too long to be processed by the LLM, we must chunk it, and do one pass through the LLM per chunk.

In [23]:
def chunk_lines(lines, max_chars=4000, max_lines=40):
    """
    Split a list of lines into ordered chunks, constrained by both
    maximum number of characters and maximum number of lines.
    """
    chunks = []
    current_chunk = []
    current_chars = 0

    for line in lines:
        line = "" if line is None else str(line)
        line_len = len(line) + 1  # account for newline

        if current_chunk and (
            len(current_chunk) >= max_lines or
            current_chars + line_len > max_chars
        ):
            chunks.append(current_chunk)
            current_chunk = []
            current_chars = 0

        current_chunk.append(line)
        current_chars += line_len

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

In [24]:
def correct_chunks_with_llm(chunks, llm, system_prompt, temperature=0.0):
    corrected_chunks = []

    for i, chunk in enumerate(chunks, 1):
        chunk_text = "\n".join(chunk)

        response = llm.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Texto OCR:\n\n{chunk_text}"}
            ],
            temperature=temperature
        )

        corrected_text = response["choices"][0]["message"]["content"].strip()
        corrected_chunks.append(corrected_text)

        print(f"Processed chunk {i}/{len(chunks)}")

    return corrected_chunks

In [25]:
# Ordered OCR lines
ocr_lines = raw_predictions

# Build chunks
chunks = chunk_lines(ocr_lines, max_chars=4000, max_lines=40)

print(f"Number of chunks: {len(chunks)}")

# Correct each chunk
corrected_chunks = correct_chunks_with_llm(
    chunks=chunks,
    llm=llm,
    system_prompt=SYSTEM_PROMPT,
    temperature=0.2
)

# Reconstruct full corrected text
corrected_text = "\n".join(corrected_chunks)

Number of chunks: 3


Llama.generate: 186 prefix-match hit, remaining 931 prompt tokens to eval
llama_perf_context_print:        load time =   20822.33 ms
llama_perf_context_print: prompt eval time =   29925.40 ms /   931 tokens (   32.14 ms per token,    31.11 tokens per second)
llama_perf_context_print:        eval time =   80404.00 ms /   906 runs   (   88.75 ms per token,    11.27 tokens per second)
llama_perf_context_print:       total time =  112492.12 ms /  1837 tokens
llama_perf_context_print:    graphs reused =        876
Llama.generate: 186 prefix-match hit, remaining 1297 prompt tokens to eval


Processed chunk 1/3


llama_perf_context_print:        load time =   20822.33 ms
llama_perf_context_print: prompt eval time =   32459.06 ms /  1297 tokens (   25.03 ms per token,    39.96 tokens per second)
llama_perf_context_print:        eval time =  104974.60 ms /  1127 runs   (   93.15 ms per token,    10.74 tokens per second)
llama_perf_context_print:       total time =  140266.70 ms /  2424 tokens
llama_perf_context_print:    graphs reused =       1091
Llama.generate: 186 prefix-match hit, remaining 1042 prompt tokens to eval


Processed chunk 2/3


llama_perf_context_print:        load time =   20822.33 ms
llama_perf_context_print: prompt eval time =   21474.75 ms /  1042 tokens (   20.61 ms per token,    48.52 tokens per second)
llama_perf_context_print:        eval time =   45415.81 ms /   499 runs   (   91.01 ms per token,    10.99 tokens per second)
llama_perf_context_print:       total time =   67881.29 ms /  1541 tokens
llama_perf_context_print:    graphs reused =        483


Processed chunk 3/3


### Global text reconstruction for evaluation

In this step, we reconstruct the full text for both the ground truth and the model outputs in order to perform evaluation at the document level. This is done for both raw OCR predictions and LLM post-processed outputs, enabling a fair and objective comparison of their performance.

During this comparison, we observe that although the LLM output is of higher quality and readability, it does not aim to preserve a strict character-by-character reconstruction of the original text. Instead, it may introduce small reformulations or corrections that improve fluency but deviate from the exact transcription. As a consequence, traditional OCR metrics such as CER and WER may not fully capture these improvements, and can sometimes report similar or even worse results for the LLM output.

To better evaluate this behavior, we introduce more permissive, word-level similarity metrics that focus on vocabulary overlap rather than exact character alignment.

---

### Jaccard Similarity

Jaccard similarity measures the overlap between the sets of unique words in the predicted and reference texts:

$$
J(A, B) = \frac{|A \cap B|}{|A \cup B|}
$$

where \(A\) and \(B\) are the sets of words in the prediction and ground truth, respectively.

**Interpretation:**  
- A value of 1 indicates identical vocabularies.  
- A value of 0 indicates no shared words.  

This metric ignores word order and frequency, focusing only on whether the same words appear in both texts.

**Relevance to OCR and LLM:**  
OCR outputs often contain distorted tokens (e.g., *“honbres”* instead of *“hombres”*), which are treated as completely different words. In contrast, LLM-based post-processing can recover valid words, increasing the overlap with the ground truth and therefore improving the Jaccard score.

---

### TF-IDF Cosine Similarity

TF-IDF cosine similarity compares the texts based on weighted word importance:

$$
\text{similarity}(A, B) = \frac{A \cdot B}{\|A\| \, \|B\|}
$$

where \(A\) and \(B\) are TF-IDF vector representations of the texts.

- **TF (Term Frequency):** how often a word appears in a document  
- **IDF (Inverse Document Frequency):** how informative a word is across documents  

**Interpretation:**  
- Values close to 1 indicate strong similarity in important vocabulary.  
- Values close to 0 indicate little shared content.  

**Relevance to OCR and LLM:**  
This metric rewards overlap in meaningful words while reducing the impact of common words (e.g., *“de”*, *“el”*). Since LLM post-processing tends to recover correct and meaningful words from noisy OCR outputs, it is expected to improve TF-IDF cosine similarity, even when strict transcription metrics do not.

---

In summary, while CER and WER measure transcription fidelity, Jaccard similarity and TF-IDF cosine similarity provide complementary insights into vocabulary recovery and overall textual coherence. These metrics are therefore particularly useful for evaluating the impact of LLM-based post-processing in scenarios where exact character-level alignment is not strictly preserved.

In [26]:
gt_lines = [row["gt_text"] for row in rows]
gt_text = "\n".join(gt_lines)

gt_global = normalize_global_text(gt_text, lowercase=False, remove_eol_hyphen=True)
llm_global = normalize_global_text(corrected_text, lowercase=False, remove_eol_hyphen=True)
ocr_global = normalize_global_text("\n".join(raw_predictions), lowercase=False, remove_eol_hyphen=True)

In [27]:
raw_global_metrics = compute_metrics(
    predictions=[ocr_global],
    references=[gt_global],
    strip_extra_spaces=True,
)

llm_global_metrics = compute_metrics(
    predictions=[llm_global],
    references=[gt_global],
    strip_extra_spaces=True,
)

print_metrics("Raw OCR global results", raw_global_metrics, show_ned=False)
print()
print_metrics("LLM post-processed global results", llm_global_metrics, show_ned=False)

Raw OCR global results
----------------------
Samples:                    1
CER:                        0.1783
Character accuracy:         0.8217
WER:                        0.4839

LLM post-processed global results
---------------------------------
Samples:                    1
CER:                        0.3126
Character accuracy:         0.6874
WER:                        0.5070


In [28]:
raw_jaccard = jaccard_similarity(ocr_global, gt_global)
llm_jaccard = jaccard_similarity(llm_global, gt_global)

raw_tfidf_cos = tfidf_cosine_similarity(ocr_global, gt_global)
llm_tfidf_cos = tfidf_cosine_similarity(llm_global, gt_global)

print("Global similarity metrics")
print("-------------------------")
print(f"Raw OCR  - Jaccard similarity:       {raw_jaccard:.4f}")
print(f"Raw OCR  - TF-IDF cosine similarity: {raw_tfidf_cos:.4f}")
print()
print(f"LLM post - Jaccard similarity:       {llm_jaccard:.4f}")
print(f"LLM post - TF-IDF cosine similarity: {llm_tfidf_cos:.4f}")

Global similarity metrics
-------------------------
Raw OCR  - Jaccard similarity:       0.2851
Raw OCR  - TF-IDF cosine similarity: 0.9443

LLM post - Jaccard similarity:       0.3106
LLM post - TF-IDF cosine similarity: 0.9523
